# Beyond NBR and NDVI: full pipeline on Colab

Runtime -> Change runtime type -> GPU. Outputs are copied to Google Drive so a disconnect does not lose them.

In [ ]:
REPO = "https://github.com/inayatarshad/beyond-spectral-indices.git"
!git clone $REPO repo
%cd repo
!pip install -q -e ".[forest]"

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/bsi_runs

## 1. Check the CaBuAr file layout once
If no events are found in step 2, paste this output into the chat and adjust `src/bsi/data/cabuar.py`.

In [ ]:
!python scripts/prepare_data.py configs/burn.yaml --inspect

## 2. Burn scars

In [ ]:
!python scripts/prepare_data.py configs/burn.yaml
!python scripts/fit_index.py configs/burn.yaml
for v in ["lgbm_index", "lgbm_all", "unet_index", "unet_all"]:
    !python scripts/train.py configs/burn.yaml {v}
!python scripts/evaluate.py configs/burn.yaml
!python scripts/explain.py configs/burn.yaml
!cp -r runs/burn /content/drive/MyDrive/bsi_runs/

## 3. Forest

In [ ]:
!python scripts/prepare_data.py configs/forest.yaml
!python scripts/fit_index.py configs/forest.yaml
for v in ["lgbm_index", "lgbm_all", "unet_index", "unet_all"]:
    !python scripts/train.py configs/forest.yaml {v}
!python scripts/evaluate.py configs/forest.yaml
!python scripts/explain.py configs/forest.yaml
!cp -r runs/forest /content/drive/MyDrive/bsi_runs/

## 4. Paper tables and figures

In [ ]:
!python scripts/make_paper_assets.py configs/burn.yaml configs/forest.yaml
!cp -r paper /content/drive/MyDrive/bsi_runs/